In [1]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import plotly.express as px
import plotly.graph_objects as go
import warnings
import subprocess
import time
import os
import time
import sqlite3
import glob
import time

warnings.filterwarnings('ignore')

caminho_absoluto = r'C:\Users\lucas\repositorios\mestrado_luedsbr\SRC\analista_cenario'
sys.path.append(caminho_absoluto)

try:
    from analistaContigencia import *
except ModuleNotFoundError as e:
    print(f" Erro: {e}")

# =============================================================================
# CONFIGURAÇÃO
# =============================================================================

TOTAL_CEN = 100
script_julia = r"C:\Users\lucas\repositorios\mestrado_luedsbr\SRC\modelos_matematicos\fluxPotContigencia.jl"

# Execução Julia

In [2]:
# 1. EXECUÇÃO DOS CENÁRIOS JULIA (SIMPLES)
# =============================================================================

def executar_cenarios_julia(total_cenarios, script_path):
    print(f"🎯 Executando {total_cenarios} cenários Julia...")
    
    tempo_inicio = time.time()
    sucessos = 0
    
    for i in range(total_cenarios):
        resultado = subprocess.run(
            ['julia', script_path],
            capture_output=True,
            text=True,
            encoding='utf-8',
            errors='ignore'
        )
        
        if resultado.returncode == 0:
            print("█", end="", flush=True)
            sucessos += 1
        else:
            print("❌", end="", flush=True)
            if resultado.stderr:
                linhas = resultado.stderr.split('\n')
                for linha in linhas:
                    if 'Error' in linha or 'ERROR' in linha:
                        print(f"\nERRO: {linha}")
                        break
    
    tempo_total = time.time() - tempo_inicio
    print(f"\n✅ Concluído! {sucessos}/{total_cenarios} sucessos")
    return tempo_total

# EXECUTAR
tempo_execucao = executar_cenarios_julia(TOTAL_CEN, script_julia)

🎯 Executando 100 cenários Julia...
████████████████████████████████████████████████████████████████████████████████████████████████████
✅ Concluído! 100/100 sucessos


# SQL

In [3]:
# 2. CONSOLIDAR RESULTADOS
# =============================================================================

def consolidar_resultados():
    print("📊 Consolidando resultados...")
    
    # Encontrar todos os bancos de dados
    bancos = glob.glob("resultados_opf_contingencias*.db")
    print(f"Encontrados {len(bancos)} bancos de dados")
    
    # Criar banco consolidado
    conn_destino = sqlite3.connect("resultados_consolidados.db")
    
    total_registros = 0
    for banco in bancos:
        try:
            conn_origem = sqlite3.connect(banco)
            
            # Copiar todas as tabelas
            for tabela in ['execucoes', 'contingencias', 'geradores_contingencia', 
                          'barras_contingencia', 'linhas_contingencia']:
                try:
                    df = pd.read_sql_query(f"SELECT * FROM {tabela}", conn_origem)
                    if len(df) > 0:
                        df.to_sql(tabela, conn_destino, if_exists='append', index=False)
                        total_registros += len(df)
                except:
                    continue
            
            conn_origem.close()
        except:
            continue
    
    conn_destino.close()
    print(f"✅ Consolidados {total_registros} registros")
    return total_registros

# CONSOLIDAR
total_registros = consolidar_resultados()

📊 Consolidando resultados...
Encontrados 1 bancos de dados
✅ Consolidados 5244 registros


In [4]:
# 3. CARREGAR DADOS PARA ANÁLISE
# =============================================================================

def carregar_dados_consolidados():
    print("📈 Carregando dados consolidados...")
    
    conn = sqlite3.connect("resultados_consolidados.db")
    
    dados = {}
    tabelas = ['execucoes', 'contingencias', 'geradores_contingencia', 
               'barras_contingencia', 'linhas_contingencia']
    
    for tabela in tabelas:
        try:
            dados[tabela] = pd.read_sql_query(f"SELECT * FROM {tabela}", conn)
            print(f"✅ {tabela}: {len(dados[tabela])} registros")
        except:
            dados[tabela] = pd.DataFrame()
            print(f"❌ {tabela}: não disponível")
    
    conn.close()
    return dados

# CARREGAR DADOS
dados = carregar_dados_consolidados()

📈 Carregando dados consolidados...
✅ execucoes: 92 registros
✅ contingencias: 368 registros
✅ geradores_contingencia: 2576 registros
✅ barras_contingencia: 1104 registros
✅ linhas_contingencia: 1104 registros


# Resumo

In [5]:
# =============================================================================
# 0. ESTATÍSTICAS RESUMO
# =============================================================================

def gerar_resumo_estatistico(dados):
    """Gerar resumo estatístico final"""
    print("\n" + "="*60)
    print("📈 RESUMO ESTATÍSTICO FINAL")
    print("="*60)
    
    if 'contingencias' in dados and len(dados['contingencias']) > 0:
        df = dados['contingencias']
        
        print(f"📊 Total de contingências analisadas: {len(df)}")
        print(f"🔢 Total de execuções únicas: {df['id_execucao'].nunique()}")
        
        print(f"💡 Demanda média: {df['total_carga_pu'].mean():.4f} pu")
        print(f"🌬️  Curtailment médio: {df['total_curtailment_pu'].mean():.4f} pu")
        print(f"⚡ Déficit médio: {df['total_deficit_pu'].mean():.4f} pu")
        print(f"💰 Custo médio: ${df['custo_total_usd_h'].mean():.2f}/h")
        
        # Contingências com déficit
        ctg_com_deficit = df[df['total_deficit_pu'] > 0.01]
        print(f"⚠️  Contingências com déficit: {len(ctg_com_deficit)}")
        
    if 'geradores_contingencia' in dados:
        geradores_gwd = dados['geradores_contingencia'][dados['geradores_contingencia']['tipo'] == 'GWD']
        if len(geradores_gwd) > 0:
            print(f"🌪️  Total de geradores eólicos: {geradores_gwd['id_gerador'].nunique()}")

# GERAR RESUMO
gerar_resumo_estatistico(dados)

print(f"\n✅ PROCESSO CONCLUÍDO!")
print(f"⏱️  Tempo de execução: {tempo_execucao:.1f} segundos")
print(f"📁 Registros consolidados: {total_registros}")
print(f"🎯 Cenários executados: {TOTAL_CEN}")


📈 RESUMO ESTATÍSTICO FINAL
📊 Total de contingências analisadas: 368
🔢 Total de execuções únicas: 92
💡 Demanda média: 1.0059 pu
🌬️  Curtailment médio: 0.0000 pu
⚡ Déficit médio: 0.0018 pu
💰 Custo médio: $4204.03/h
⚠️  Contingências com déficit: 8
🌪️  Total de geradores eólicos: 1

✅ PROCESSO CONCLUÍDO!
⏱️  Tempo de execução: 1401.4 segundos
📁 Registros consolidados: 5244
🎯 Cenários executados: 100


# Barras

In [6]:
# 4. ANÁLISE E GRÁFICOS - VISÃO POR CONTINGÊNCIA COM GERAÇÃO POR TIPO
# =============================================================================

import plotly.express as px
import plotly.graph_objects as go
import numpy as np

def plotar_por_contingencia_com_geracao_tipo(dados):
    """Gráfico individual para cada contingência com geração agregada por tipo"""
    if 'contingencias' not in dados or len(dados['contingencias']) == 0:
        print("❌ Dados insuficientes para gráfico")
        return
    
    df_contingencias = dados['contingencias']
    
    # Buscar informações das contingências para descrição
    if 'contingencias_info' in dados:
        df_info = dados['contingencias_info']
    else:
        df_info = df_contingencias[['id_contingencia', 'descricao', 'linhas_removidas']].drop_duplicates()
    
    # Buscar dados de geração por tipo
    if 'geradores_contingencia' in dados:
        df_geradores = dados['geradores_contingencia']
        
        # Agrupar geração por tipo, execução e contingência
        geracao_agrupada = df_geradores.groupby(['id_execucao', 'id_contingencia', 'tipo'])['geracao_pu'].sum().reset_index()
        
        # Pivot para ter colunas por tipo
        geracao_pivot = geracao_agrupada.pivot_table(
            index=['id_execucao', 'id_contingencia'],
            columns='tipo',
            values='geracao_pu',
            fill_value=0
        ).reset_index()
    else:
        geracao_pivot = pd.DataFrame()
    
    # Ordenar contingências numericamente
    contingencias_unicas = sorted(df_contingencias['id_contingencia'].unique(), 
                                 key=lambda x: int(x[3:]) if x[3:].isdigit() else x)
    
    # Criar um gráfico para cada contingência
    for contingencia in contingencias_unicas:
        df_ctg = df_contingencias[df_contingencias['id_contingencia'] == contingencia]
        
        # Buscar informações desta contingência
        info_ctg = df_info[df_info['id_contingencia'] == contingencia].iloc[0] if len(df_info[df_info['id_contingencia'] == contingencia]) > 0 else None
        
        # Ordenar execuções numericamente
        df_ctg = df_ctg.sort_values('id_execucao')
        df_ctg['execucao_num'] = range(1, len(df_ctg) + 1)
        
        # Juntar com dados de geração por tipo
        if not geracao_pivot.empty:
            df_ctg_com_geracao = df_ctg.merge(
                geracao_pivot[geracao_pivot['id_contingencia'] == contingencia],
                on=['id_execucao', 'id_contingencia'],
                how='left'
            )
        else:
            df_ctg_com_geracao = df_ctg.copy()
            # Adicionar colunas vazias para os tipos
            for tipo in ['UTE', 'UTH', 'GWD']:
                df_ctg_com_geracao[tipo] = 0
        
        # Preencher NaN com 0
        for tipo in ['UTE', 'UTH', 'GWD']:
            if tipo in df_ctg_com_geracao.columns:
                df_ctg_com_geracao[tipo] = df_ctg_com_geracao[tipo].fillna(0)
        
        # Identificar cenários com corte de carga
        cenarios_com_corte = df_ctg_com_geracao[df_ctg_com_geracao['total_deficit_pu'] > 0.001]
        
        fig = go.Figure()
        
        # Demanda (linha tracejada)
        fig.add_trace(go.Scatter(
            x=df_ctg_com_geracao['execucao_num'],
            y=df_ctg_com_geracao['total_carga_pu'],
            name='Demanda',
            mode='lines',
            line=dict(color='black', width=3, dash='dash')
        ))
        
        # Geração Eólica (GWD)
        if 'GWD' in df_ctg_com_geracao.columns:
            fig.add_trace(go.Scatter(
                x=df_ctg_com_geracao['execucao_num'],
                y=df_ctg_com_geracao['GWD'],
                name='Geração Eólica',
                mode='markers+lines',
                marker=dict(color='green', size=8),
                line=dict(color='green', width=2)
            ))
        
        # Geração Hidrelétrica (UTH)
        if 'UTH' in df_ctg_com_geracao.columns:
            fig.add_trace(go.Scatter(
                x=df_ctg_com_geracao['execucao_num'],
                y=df_ctg_com_geracao['UTH'],
                name='Geração Hidrelétrica',
                mode='markers+lines',
                marker=dict(color='blue', size=8, symbol='square'),
                line=dict(color='blue', width=2)
            ))
        
        # Geração Térmica (UTE)
        if 'UTE' in df_ctg_com_geracao.columns:
            fig.add_trace(go.Scatter(
                x=df_ctg_com_geracao['execucao_num'],
                y=df_ctg_com_geracao['UTE'],
                name='Geração Térmica',
                mode='markers+lines',
                marker=dict(color='red', size=8, symbol='diamond'),
                line=dict(color='red', width=2)
            ))
        
        # Curtailment
        fig.add_trace(go.Scatter(
            x=df_ctg_com_geracao['execucao_num'],
            y=df_ctg_com_geracao['total_curtailment_pu'],
            name='Curtailment',
            mode='markers+lines',
            marker=dict(color='orange', size=6, symbol='triangle-up'),
            line=dict(color='orange', width=1, dash='dot')
        ))
        
        # Destacar cenários com corte de carga
        if len(cenarios_com_corte) > 0:
            fig.add_trace(go.Scatter(
                x=cenarios_com_corte['execucao_num'],
                y=cenarios_com_corte['total_deficit_pu'],
                name='CORTE DE CARGA ',
                mode='markers',
                marker=dict(color='black', size=12, symbol='x', line=dict(width=2, color='white')),
                line=dict(color='black', width=3)
            ))
        
        # Criar título com informações da contingência
        titulo = f'Contingência: {contingencia}'
        if info_ctg is not None:
            descricao = info_ctg['descricao'] if pd.notna(info_ctg['descricao']) else "Sem descrição"
            linhas_removidas = info_ctg['linhas_removidas'] if pd.notna(info_ctg['linhas_removidas']) else "Nenhuma"
            titulo += f'<br><span style="font-size:12px">{descricao}</span>'
            titulo += f'<br><span style="font-size:10px">Linhas removidas: {linhas_removidas}</span>'
        
        fig.update_layout(
            title=titulo,
            xaxis_title='Número da Execução',
            yaxis_title='Potência (pu)',
            height=500,
            showlegend=True
        )
        
        fig.show()
        
        # Relatório detalhado da contingência
        print(f"\n📊 RELATÓRIO - {contingencia}:")
        print(f"   Execuções analisadas: {len(df_ctg_com_geracao)}")
        
        # Estatísticas de geração por tipo
        if not geracao_pivot.empty:
            print(f"   Geração Média por Tipo:")
            for tipo in ['GWD', 'UTH', 'UTE']:
                if tipo in df_ctg_com_geracao.columns:
                    media = df_ctg_com_geracao[tipo].mean()
                    if media > 0:
                        print(f"      {tipo}: {media:.4f} pu")
        
        print(f"   Demanda média: {df_ctg_com_geracao['total_carga_pu'].mean():.4f} pu")
        print(f"   Curtailment médio: {df_ctg_com_geracao['total_curtailment_pu'].mean():.4f} pu")
        
        if len(cenarios_com_corte) > 0:
            print(f"    Cenários com corte de carga: {len(cenarios_com_corte)}")
            print(f"      Execuções: {list(cenarios_com_corte['execucao_num'])}")
        else:
            print(f"   ✅ Nenhum corte de carga identificado")
    
    return df_ctg_com_geracao

def plotar_corte_carga_barras(dados):
    """Gráfico de corte de carga médio por barra"""
    if 'barras_contingencia' not in dados:
        print("❌ Dados de barras insuficientes")
        return
    
    df_barras = dados['barras_contingencia']
    
    # Calcular corte de carga médio por barra
    corte_por_barra = df_barras.groupby(['id_barra', 'tipo']).agg({
        'deficit_pu': ['mean', 'max', 'count']
    }).round(6).reset_index()
    
    corte_por_barra.columns = ['id_barra', 'tipo_barra', 'deficit_medio', 'deficit_maximo', 'num_cenarios']
    
    # Filtrar apenas barras com corte de carga
    barras_com_corte = corte_por_barra[corte_por_barra['deficit_medio'] > 0]
    
    if len(barras_com_corte) == 0:
        print("✅ Nenhum corte de carga identificado nas barras")
        return
    
    # Ordenar por maior corte médio
    barras_com_corte = barras_com_corte.sort_values('deficit_medio', ascending=False)
    
    print("🏆 BARRAS COM MAIOR CORTE DE CARGA MÉDIO:")
    print(barras_com_corte[['id_barra', 'tipo_barra', 'deficit_medio', 'num_cenarios']].head(10).to_string(index=False))
    
    # Gráfico de barras
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        x=barras_com_corte['id_barra'].head(10),
        y=barras_com_corte['deficit_medio'].head(10),
        marker_color='crimson',
        text=barras_com_corte['deficit_medio'].head(10).round(4),
        textposition='auto',
        hovertemplate='<b>Barra %{x}</b><br>Corte médio: %{y:.4f} pu<br>Tipo: %{customdata}<br>Cenários: %{text}',
        customdata=barras_com_corte['tipo_barra'].head(10)
    ))
    
    fig.update_layout(
        title='Corte de Carga Médio por Barra (Top 10)',
        xaxis_title='Barra',
        yaxis_title='Corte de Carga Médio (pu)',
        height=500
    )
    
    fig.show()
    
    return barras_com_corte

def plotar_curtailment_por_contingencia_ordenado(dados):
    """Gráfico de curtailment médio por contingência ordenado"""
    if 'contingencias' not in dados:
        return
    
    df_contingencias = dados['contingencias']
    
    # Ordenar contingências numericamente
    contingencias_ordenadas = sorted(df_contingencias['id_contingencia'].unique(), 
                                    key=lambda x: int(x[3:]) if x[3:].isdigit() else x)
    
    # Calcular curtailment médio por contingência
    curtailment_por_ctg = df_contingencias.groupby('id_contingencia').agg({
        'total_curtailment_pu': ['mean', 'std', 'max', 'count']
    }).round(6).reset_index()
    
    curtailment_por_ctg.columns = ['id_contingencia', 'curtailment_medio', 'curtailment_std', 'curtailment_max', 'num_cenarios']
    
    # Reordenar conforme a ordem numérica
    curtailment_por_ctg['ordem'] = curtailment_por_ctg['id_contingencia'].apply(
        lambda x: int(x[3:]) if x[3:].isdigit() else 999
    )
    curtailment_por_ctg = curtailment_por_ctg.sort_values('ordem')
    
    print("🌬️  CURTAILMENT MÉDIO POR CONTINGÊNCIA:")
    print(curtailment_por_ctg[['id_contingencia', 'curtailment_medio', 'curtailment_std', 'num_cenarios']].to_string(index=False))
    
    # Gráfico de curtailment médio
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        x=curtailment_por_ctg['id_contingencia'],
        y=curtailment_por_ctg['curtailment_medio'],
        marker_color='orange',
        error_y=dict(
            type='data',
            array=curtailment_por_ctg['curtailment_std'],
            visible=True
        ),
        text=curtailment_por_ctg['curtailment_medio'].round(4),
        textposition='auto',
        hovertemplate='<b>%{x}</b><br>Curtailment médio: %{y:.4f} pu<br>Desvio: ±%{customdata:.4f} pu',
        customdata=curtailment_por_ctg['curtailment_std']
    ))
    
    fig.update_layout(
        title='Curtailment Médio por Tipo de Contingência',
        xaxis_title='Contingência',
        yaxis_title='Curtailment Médio (pu)',
        height=500
    )
    
    fig.show()
    
    return curtailment_por_ctg



In [7]:
# =============================================================================
# EXECUÇÃO DOS GRÁFICOS
# =============================================================================

print("="*60)
print("ANÁLISE COMPLETA - GERAÇÃO POR TIPO E CONTINGÊNCIAS")
print("="*60)

# 1. Gráficos por contingência com geração por tipo
print("\n1️⃣  GRÁFICOS POR CONTINGÊNCIA COM GERAÇÃO POR TIPO")
df_por_ctg = plotar_por_contingencia_com_geracao_tipo(dados)

# 2. Análise de corte de carga por barra
print("\n2️⃣  CORTE DE CARGA POR BARRA")
corte_barras = plotar_corte_carga_barras(dados)

# 3. Curtailment médio por contingência
print("\n3️⃣  CURTAILMENT MÉDIO POR CONTINGÊNCIA")
curtailment_stats = plotar_curtailment_por_contingencia_ordenado(dados)



ANÁLISE COMPLETA - GERAÇÃO POR TIPO E CONTINGÊNCIAS

1️⃣  GRÁFICOS POR CONTINGÊNCIA COM GERAÇÃO POR TIPO



📊 RELATÓRIO - CTG-1:
   Execuções analisadas: 92
   Geração Média por Tipo:
      GWD: 0.4471 pu
      UTH: 0.5589 pu
   Demanda média: 1.0059 pu
   Curtailment médio: 0.0000 pu
   ✅ Nenhum corte de carga identificado



📊 RELATÓRIO - CTG-2:
   Execuções analisadas: 92
   Geração Média por Tipo:
      GWD: 0.4471 pu
      UTH: 0.5576 pu
      UTE: 0.0013 pu
   Demanda média: 1.0059 pu
   Curtailment médio: 0.0000 pu
   ✅ Nenhum corte de carga identificado



📊 RELATÓRIO - CTG-3:
   Execuções analisadas: 92
   Geração Média por Tipo:
      GWD: 0.4471 pu
      UTH: 0.5427 pu
      UTE: 0.0105 pu
   Demanda média: 1.0059 pu
   Curtailment médio: 0.0000 pu
    Cenários com corte de carga: 6
      Execuções: [15, 31, 42, 57, 78, 87]



📊 RELATÓRIO - CTG-4:
   Execuções analisadas: 92
   Geração Média por Tipo:
      GWD: 0.4471 pu
      UTH: 0.5576 pu
   Demanda média: 1.0059 pu
   Curtailment médio: 0.0000 pu
    Cenários com corte de carga: 2
      Execuções: [31, 57]

2️⃣  CORTE DE CARGA POR BARRA
🏆 BARRAS COM MAIOR CORTE DE CARGA MÉDIO:
id_barra tipo_barra  deficit_medio  num_cenarios
       3         PQ       0.001755           368



3️⃣  CURTAILMENT MÉDIO POR CONTINGÊNCIA
🌬️  CURTAILMENT MÉDIO POR CONTINGÊNCIA:
id_contingencia  curtailment_medio  curtailment_std  num_cenarios
          CTG-1                0.0              0.0            92
          CTG-2                0.0              0.0            92
          CTG-3                0.0              0.0            92
          CTG-4                0.0              0.0            92


# Linhas

In [8]:
def plotar_uso_capacidade_linhas(dados):
    """Gráfico de uso da capacidade das linhas de transmissão por contingência"""
    if 'linhas_contingencia' not in dados:
        print("❌ Dados de linhas insuficientes")
        return
    
    df_linhas = dados['linhas_contingencia']
    
    # Buscar informações das contingências para descrição
    if 'contingencias_info' in dados:
        df_info = dados['contingencias_info']
    else:
        df_contingencias = dados.get('contingencias', pd.DataFrame())
        df_info = df_contingencias[['id_contingencia', 'descricao', 'linhas_removidas']].drop_duplicates()
    
    # Ordenar contingências numericamente
    contingencias_unicas = sorted(df_linhas['id_contingencia'].unique(), 
                                 key=lambda x: int(x[3:]) if x[3:].isdigit() else x)
    
    # Criar um gráfico para cada contingência
    for contingencia in contingencias_unicas:
        df_linhas_ctg = df_linhas[df_linhas['id_contingencia'] == contingencia]
        
        # Buscar informações desta contingência
        info_ctg = df_info[df_info['id_contingencia'] == contingencia].iloc[0] if len(df_info[df_info['id_contingencia'] == contingencia]) > 0 else None
        
        # Ordenar execuções numericamente
        execucoes_unicas = sorted(df_linhas_ctg['id_execucao'].unique())
        df_linhas_ctg['execucao_num'] = df_linhas_ctg['id_execucao'].map(
            {exec: i+1 for i, exec in enumerate(execucoes_unicas)}
        )
        
        # Agrupar por linha para análise
        linhas_agrupadas = df_linhas_ctg.groupby('id_linha').agg({
            'utilizacao_percentual': ['mean', 'max', 'std'],
            'id_barra_origem': 'first',
            'id_barra_destino': 'first',
            'limite_pu': 'first'
        }).round(2).reset_index()
        
        linhas_agrupadas.columns = ['id_linha', 'util_media', 'util_maxima', 'util_std', 'origem', 'destino', 'limite_pu']
        
        # Top 10 linhas com maior utilização média
        top_linhas = linhas_agrupadas.nlargest(10, 'util_media')
        
        fig = go.Figure()
        
        # Linhas de limite crítico
        fig.add_hline(y=80, line_dash="dash", line_color="orange", 
                     annotation_text="Limite 80% - Atenção", annotation_position="right")
        fig.add_hline(y=95, line_dash="dash", line_color="red",
                     annotation_text="Limite 95% - Crítico", annotation_position="right")
        
        # Para cada linha top, plotar a utilização em cada execução
        for _, linha_info in top_linhas.iterrows():
            linha_id = linha_info['id_linha']
            df_linha = df_linhas_ctg[df_linhas_ctg['id_linha'] == linha_id]
            
            # Ordenar por execução
            df_linha = df_linha.sort_values('execucao_num')
            
            fig.add_trace(go.Scatter(
                x=df_linha['execucao_num'],
                y=df_linha['utilizacao_percentual'],
                name=f'{linha_id} ({linha_info["origem"]}-{linha_info["destino"]})',
                mode='markers+lines',
                marker=dict(size=6),
                line=dict(width=2),
                hovertemplate='<b>Linha %{customdata[0]}</b><br>Execução: %{x}<br>Utilização: %{y:.1f}%<br>Origem: %{customdata[1]}<br>Destino: %{customdata[2]}<br>Limite: %{customdata[3]:.3f} pu',
                customdata=np.column_stack([
                    [linha_id] * len(df_linha),
                    df_linha['id_barra_origem'],
                    df_linha['id_barra_destino'],
                    df_linha['limite_pu']
                ])
            ))
        
        # Criar título com informações da contingência
        titulo = f'Uso da Capacidade das Linhas - Contingência: {contingencia}'
        if info_ctg is not None:
            descricao = info_ctg['descricao'] if pd.notna(info_ctg['descricao']) else "Sem descrição"
            linhas_removidas = info_ctg['linhas_removidas'] if pd.notna(info_ctg['linhas_removidas']) else "Nenhuma"
            titulo += f'<br><span style="font-size:12px">{descricao}</span>'
            titulo += f'<br><span style="font-size:10px">Linhas removidas: {linhas_removidas}</span>'
        
        fig.update_layout(
            title=titulo,
            xaxis_title='Número da Execução',
            yaxis_title='Utilização da Capacidade (%)',
            height=500,
            showlegend=True,
            legend=dict(
                orientation="v",
                yanchor="top",
                y=0.99,
                xanchor="left",
                x=1.02
            )
        )
        
        fig.show()
        
        # Relatório detalhado da contingência
        print(f"\n📊 RELATÓRIO DE LINHAS - {contingencia}:")
        print(f"   Execuções analisadas: {len(execucoes_unicas)}")
        print(f"   Linhas monitoradas: {len(df_linhas_ctg['id_linha'].unique())}")
        
        # Linhas críticas (acima de 80%)
        linhas_criticas = linhas_agrupadas[linhas_agrupadas['util_media'] > 80]
        linhas_muito_criticas = linhas_agrupadas[linhas_agrupadas['util_media'] > 95]
        
        print(f"   ⚠️  Linhas com utilização média >80%: {len(linhas_criticas)}")
        print(f"   🔴 Linhas com utilização média >95%: {len(linhas_muito_criticas)}")
        
        if len(linhas_criticas) > 0:
            print(f"   Top linhas críticas:")
            for _, linha in linhas_criticas.head(5).iterrows():
                print(f"      {linha['id_linha']}: {linha['util_media']:.1f}% (máx: {linha['util_maxima']:.1f}%)")
    
    return df_linhas

def plotar_linhas_mais_criticas_geral(dados):
    """Gráfico das linhas mais críticas em todas as contingências"""
    if 'linhas_contingencia' not in dados:
        return
    
    df_linhas = dados['linhas_contingencia']
    
    # Calcular estatísticas gerais por linha
    stats_linhas = df_linhas.groupby(['id_linha', 'id_barra_origem', 'id_barra_destino']).agg({
        'utilizacao_percentual': ['mean', 'max', 'std', 'count'],
        'limite_pu': 'first'
    }).round(2).reset_index()
    
    stats_linhas.columns = ['id_linha', 'origem', 'destino', 'util_media', 'util_maxima', 'util_std', 'num_medicoes', 'limite_pu']
    
    # Top 15 linhas com maior utilização média
    top_linhas_geral = stats_linhas.nlargest(10, 'util_media')
    
    print("🏆 LINHAS MAIS CRÍTICAS - VISÃO GERAL:")
    print(top_linhas_geral[['id_linha', 'origem', 'destino', 'util_media', 'util_maxima']].to_string(index=False))
    
    # Gráfico de barras
    fig = go.Figure()
    
    # Barras para utilização média
    fig.add_trace(go.Bar(
        x=top_linhas_geral['id_linha'],
        y=top_linhas_geral['util_media'],
        name='Utilização Média',
        marker_color='lightcoral',
        text=top_linhas_geral['util_media'].round(1),
        textposition='auto'
    ))
    
    # Barras para utilização máxima
    fig.add_trace(go.Bar(
        x=top_linhas_geral['id_linha'],
        y=top_linhas_geral['util_maxima'],
        name='Utilização Máxima',
        marker_color='darkred',
        text=top_linhas_geral['util_maxima'].round(1),
        textposition='auto'
    ))
    
    # Linhas de limite
    fig.add_hline(y=80, line_dash="dash", line_color="orange")
    fig.add_hline(y=95, line_dash="dash", line_color="red")
    
    fig.update_layout(
        title='Utilização da Capacidade (Média e Máxima)',
        xaxis_title='Linha',
        yaxis_title='Utilização da Capacidade (%)',
        barmode='group',
        height=500,
        showlegend=True
    )
    
    fig.show()
    
    return top_linhas_geral

def plotar_coeficientes_lagrange_medio(dados):
    """Gráfico dos coeficientes de Lagrange médios das variáveis duais"""
    if 'linhas_contingencia' not in dados:
        print("❌ Dados de linhas insuficientes")
        return
    
    df_linhas = dados['linhas_contingencia']
    
    # Verificar se as colunas de lambda_flow existem
    if 'lambda_flow_pos' not in df_linhas.columns or 'lambda_flow_neg' not in df_linhas.columns:
        print("⚠️  Colunas de variáveis duais (lambda_flow) não encontradas nos dados")
        return
    
    # Calcular lambda_flow médio (soma dos valores absolutos)
    df_linhas['coeficiente_lagrange'] = df_linhas['lambda_flow_pos'].abs() + df_linhas['lambda_flow_neg'].abs()
    

    top_linhas = df_linhas.groupby('id_linha').agg({
        'coeficiente_lagrange': ['mean', 'std', 'max'],
        'id_barra_origem': 'first',
        'id_barra_destino': 'first'
    }).round(6)
    
    top_linhas.columns = ['coef_medio', 'coef_std', 'coef_max', 'origem', 'destino']
    top_linhas = top_linhas.nlargest(10, 'coef_medio')
    
    print("📊 LINHAS COM MAIORES COEFICIENTES DE LAGRANGE (VARIÁVEIS DUAIS):")
    print(top_linhas[['coef_medio', 'coef_std', 'coef_max', 'origem', 'destino']].round(6))
    
    # Gráfico
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        x=top_linhas.index,
        y=top_linhas['coef_medio'],
        marker_color='purple',
        error_y=dict(
            type='data',
            array=top_linhas['coef_std'],
            visible=True
        ),
        text=top_linhas['coef_medio'].round(4),
        textposition='auto',
        hovertemplate='<b>Linha %{x}</b><br>Lambda médio: %{y:.6f}<br>Origem: %{customdata[0]}<br>Destino: %{customdata[1]}',
        customdata=top_linhas[['origem', 'destino']].values
    ))
    
    fig.update_layout(
        title='Variáveis Duais de Fluxo Médias',
        xaxis_title='Linha',
        yaxis_title='Valor da Variável Dual',
        height=500
    )
    
    fig.show()
    
    return top_linhas

# =============================================================================
# EXECUÇÃO DOS GRÁFICOS
# =============================================================================

print("="*60)
print("ANÁLISE DE CAPACIDADE DAS LINHAS DE TRANSMISSÃO")
print("="*60)

# 1. Gráficos de uso de capacidade por contingência
print("\n1️⃣  USO DA CAPACIDADE POR CONTINGÊNCIA")
df_linhas_analise = plotar_uso_capacidade_linhas(dados)

# 2. Visão geral das linhas mais críticas
print("\n2️⃣  LINHAS MAIS CRÍTICAS - VISÃO GERAL")
linhas_criticas = plotar_linhas_mais_criticas_geral(dados)

# 3. Coeficientes de Lagrange
print("\n4️⃣  COEFICIENTES DE LAGRANGE DAS LINHAS")
lagrange_stats = plotar_coeficientes_lagrange_medio(dados)

print("\n✅ Análise de capacidade das linhas concluída!")

ANÁLISE DE CAPACIDADE DAS LINHAS DE TRANSMISSÃO

1️⃣  USO DA CAPACIDADE POR CONTINGÊNCIA



📊 RELATÓRIO DE LINHAS - CTG-1:
   Execuções analisadas: 92
   Linhas monitoradas: 3
   ⚠️  Linhas com utilização média >80%: 0
   🔴 Linhas com utilização média >95%: 0



📊 RELATÓRIO DE LINHAS - CTG-2:
   Execuções analisadas: 92
   Linhas monitoradas: 3
   ⚠️  Linhas com utilização média >80%: 0
   🔴 Linhas com utilização média >95%: 0



📊 RELATÓRIO DE LINHAS - CTG-3:
   Execuções analisadas: 92
   Linhas monitoradas: 3
   ⚠️  Linhas com utilização média >80%: 0
   🔴 Linhas com utilização média >95%: 0



📊 RELATÓRIO DE LINHAS - CTG-4:
   Execuções analisadas: 92
   Linhas monitoradas: 3
   ⚠️  Linhas com utilização média >80%: 0
   🔴 Linhas com utilização média >95%: 0

2️⃣  LINHAS MAIS CRÍTICAS - VISÃO GERAL
🏆 LINHAS MAIS CRÍTICAS - VISÃO GERAL:
id_linha origem destino  util_media  util_maxima
     1-3      1       3       35.97        100.0
     1-2      1       2       24.31        100.0
     2-3      2       3       21.94        100.0



4️⃣  COEFICIENTES DE LAGRANGE DAS LINHAS
📊 LINHAS COM MAIORES COEFICIENTES DE LAGRANGE (VARIÁVEIS DUAIS):
            coef_medio       coef_std   coef_max origem destino
id_linha                                                       
2-3       32576.086957  253377.736934  1998000.0      2       3
1-3       10869.565217  147166.940177  1999000.0      1       3
1-2          38.043478     191.561711     1000.0      1       2



✅ Análise de capacidade das linhas concluída!
